### FastAPI

In [1]:
!pip -q install fastapi uvicorn pyngrok lime

In [2]:
## ejemplo hola mundo

from fastapi import FastAPI

app = FastAPI(title="API de ejemplo (FastAPI)")

@app.get("/loqueyoquiera")
def lo_que_yo_quiera():
    return {"mensaje": "¡Hola, FastAPI!"}

@app.get("/manolo")
def manolo():
    return {"mensaje": "ADIÓS"}

@app.get("/antonio")
def antonio():
    return "la clase de febrero"


In [3]:
import threading
import uvicorn

if "server" not in globals() or server.should_exit:
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )
    server = uvicorn.Server(config)

    def run():
        server.run()

    uvicorn_thread = threading.Thread(target=run)
    uvicorn_thread.start()

print("Servidor iniciado en http://127.0.0.1:8000/docs")


Servidor iniciado en http://127.0.0.1:8000/docs


In [5]:
with open("ngrok_token.txt", "r") as file:
    ngrok_token = file.read().strip()

In [6]:
import os
from pyngrok import ngrok

ngrok.set_auth_token(ngrok_token)

try:
    ngrok.kill()
except Exception:
    pass

public_url = ngrok.connect(8000, "http")
print("URL pública:", public_url)
print("Swagger:", str(public_url).rstrip('/') + "/docs")


URL pública: NgrokTunnel: "https://karina-unpleaded-amatively.ngrok-free.dev" -> "http://localhost:8000"
Swagger: NgrokTunnel: "https://karina-unpleaded-amatively.ngrok-free.dev" -> "http://localhost:8000"/docs


In [7]:
from pyngrok import ngrok
ngrok.kill()

In [8]:
if "server" in globals():
    server.should_exit = True
    print("Servidor detenido")



Servidor detenido


#### ¿ y si tenemos un modelo de machine learning?

In [9]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import HTMLResponse
import pandas as pd
import uvicorn
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import io
import base64
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

app = FastAPI()

df = None
model = None
X_train, X_test, y_train, y_test = None, None, None, None

@app.get("/", response_class=HTMLResponse)
def read_root():
    return """
<html>
<head>
    <title>Modelo LIME & SHAP</title>
    <style>
        body { text-align: center; font-family: Arial, sans-serif; background-color: #f4f4f4; }
        .container { margin-top: 50px; }
        .btn { display: block; width: 200px; margin: 20px auto; padding: 15px; font-size: 18px; color: white; background-color: #007BFF; border: none; border-radius: 5px; text-decoration: none; }
        .btn:hover { background-color: #0056b3; }
    </style>
</head>
<body>
    <div class="container">
        <h1>Bienvenido a la API de análisis con LIME y SHAP</h1>
        <form action="/upload/" method="post" enctype="multipart/form-data">
            <input type="file" name="file" />
            <button class="btn" type="submit">Cargar Datos</button>
        </form>
        <a class="btn" href="/train">Entrenar Modelo</a>
        <a class="btn" href="/shap">Ver SHAP</a>
        <a class="btn" href="/lime">Ver LIME</a>
    </div>
</body>
</html>
    """

@app.post("/upload/")
async def upload_file(file: UploadFile = File(...)):
    global df, X_train, X_test, y_train, y_test

    # Leer el archivo subido
    contents = await file.read()
    df = pd.read_csv(io.StringIO(contents.decode('utf-8')), sep=';')

    # Procesar los datos
    categorical_features = df.select_dtypes(include="object").columns
    for col in categorical_features:
        df[col] = LabelEncoder().fit_transform(df[col])

    X = df.drop(columns=["y"])  # Asumimos que "y" es la columna objetivo
    y = df["y"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    return {"message": "Archivo cargado y procesado correctamente"}

@app.get("/train/")
def train_model():
    global model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return {"message": "Modelo entrenado correctamente"}

@app.get("/shap/", response_class=HTMLResponse)
def shap_values():
    global model, X_test
    if model is None or X_test is None:
        return "<h3>Por favor, cargue los datos y entrene el modelo antes de generar SHAP.</h3>"

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test.iloc[0:20])

    plt.figure()
    shap.summary_plot(shap_values, X_test.iloc[0:20], show=False)
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()

    return f'<img src="data:image/png;base64,{img_str}" />'

@app.get("/lime/", response_class=HTMLResponse)
def lime_explanation():
    global model, X_train, X_test
    if model is None or X_train is None or X_test is None:
        return "<h3>Por favor, cargue los datos y entrene el modelo antes de generar LIME.</h3>"

    explainer = lime.lime_tabular.LimeTabularExplainer(X_train.values, feature_names=X_train.columns, class_names=['0', '1'], discretize_continuous=True)
    exp = explainer.explain_instance(X_test.iloc[0].values, model.predict_proba, num_features=10)

    buf = io.BytesIO()
    exp.as_pyplot_figure()
    plt.savefig(buf, format='png')
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    buf.close()

    return f'<img src="data:image/png;base64,{img_str}" />'

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1050]


In [10]:
import threading
import uvicorn

if "server" not in globals() or server.should_exit:
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )
    server = uvicorn.Server(config)

    def run():
        server.run()

    uvicorn_thread = threading.Thread(target=run)
    uvicorn_thread.start()

print("Servidor iniciado en http://127.0.0.1:8000/docs")


INFO:     Started server process [1050]


Servidor iniciado en http://127.0.0.1:8000/docs


INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [11]:
import os
from pyngrok import ngrok

ngrok.set_auth_token(ngrok_token)

try:
    ngrok.kill()
except Exception:
    pass

public_url = ngrok.connect(8000, "http")
print("URL pública:", public_url)
print("Swagger:", str(public_url).rstrip('/') + "/docs")

URL pública: NgrokTunnel: "https://karina-unpleaded-amatively.ngrok-free.dev" -> "http://localhost:8000"
Swagger: NgrokTunnel: "https://karina-unpleaded-amatively.ngrok-free.dev" -> "http://localhost:8000"/docs
